# Ogham OCR: CNN+RNN Baseline (CTC)

Train a traditional CNN+RNN OCR model as a baseline comparison for TrOCR.

**Architecture:**
- Encoder: ResNet-18 (pretrained ImageNet) -> feature maps
- Decoder: 2-layer Bidirectional LSTM
- Loss: CTC (Connectionist Temporal Classification)

**Dataset:** Same 200k synthetic Ogham images from HF Hub.

**Goal:** Establish a pre-transformer baseline CER to compare against TrOCR-small (0.06%) and TrOCR-base.

**Requirements:** Colab with GPU (A100 preferred, T4 works).

---
## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q datasets huggingface_hub editdistance Pillow tqdm torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/cnn_rnn'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
# Clone repo
import os
REPO_DIR = '/content/ogham_ocr'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/newsyd04/ogham-masters.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from huggingface_hub import login
login()

---
## 2. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
HF_DATASET_NAME = 'DaraTraining/ogham-synthetic-200k'

EPOCHS = 20
BATCH_SIZE = 64          # CNN+RNN is much lighter than TrOCR
LR = 1e-3
HIDDEN_SIZE = 256        # LSTM hidden size
NUM_LSTM_LAYERS = 2
IMG_HEIGHT = 64          # Resize images to fixed height
IMG_WIDTH = 256          # Resize images to fixed width
NUM_WORKERS = 4

print(f'Config: {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}')
print(f'LSTM: {NUM_LSTM_LAYERS} layers, {HIDDEN_SIZE} hidden')
print(f'Image: {IMG_HEIGHT}x{IMG_WIDTH}')

---
## 3. Character Vocabulary (CTC)

In [ ]:
# Build character vocabulary for CTC
# Index 0 = CTC blank token
# Ogham characters: U+1681 to U+169C (28 chars) + space

OGHAM_CHARS = [chr(c) for c in range(0x1681, 0x169D)]  # 28 Ogham letters
VOCAB = ['<blank>'] + OGHAM_CHARS + [' ']  # blank + 28 chars + space = 30

char_to_idx = {c: i for i, c in enumerate(VOCAB)}
idx_to_char = {i: c for i, c in enumerate(VOCAB)}

BLANK_IDX = 0
NUM_CLASSES = len(VOCAB)

print(f'Vocabulary size: {NUM_CLASSES} (blank + {len(OGHAM_CHARS)} Ogham + space)')
print(f'Characters: {" ".join(OGHAM_CHARS)}')

---
## 4. Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torchvision import transforms
from PIL import Image

class CTCDataset(Dataset):
    """Dataset for CNN+RNN with CTC loss."""

    def __init__(self, hf_dataset, transform, char_to_idx, max_label_len=64):
        self.ds = hf_dataset
        self.transform = transform
        self.char_to_idx = char_to_idx
        self.max_label_len = max_label_len

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]

        try:
            image = sample['image'].convert('RGB')
        except Exception:
            return self.__getitem__((idx + 1) % len(self))

        image = self.transform(image)

        # Encode text as indices
        text = sample['ogham_text'].replace('\u1680', ' ')
        label = [self.char_to_idx.get(c, 0) for c in text if c in self.char_to_idx]
        label_length = len(label)

        # Pad label to fixed length
        label = label[:self.max_label_len]
        label_length = min(label_length, self.max_label_len)
        label += [0] * (self.max_label_len - len(label))

        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long),
            'label_length': torch.tensor(label_length, dtype=torch.long),
            'text': text,
        }

# Image transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Dataset class ready')

In [ ]:
# Load HF dataset
print(f'Loading: {HF_DATASET_NAME}')
hf_train = load_dataset(HF_DATASET_NAME, split='train', token=True)
hf_val = load_dataset(HF_DATASET_NAME, split='validation', token=True)
print(f'Train: {len(hf_train)}, Val: {len(hf_val)}')

train_dataset = CTCDataset(hf_train, train_transform, char_to_idx)
val_dataset = CTCDataset(hf_val, val_transform, char_to_idx)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

---
## 5. Model: CNN (ResNet-18) + BiLSTM + CTC

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class CRNN(nn.Module):
    """
    CNN+RNN model for OCR with CTC loss.

    Architecture:
        ResNet-18 (conv layers only) -> BiLSTM -> Linear -> CTC

    The CNN extracts visual features from the image.
    Features are reshaped into a sequence (one feature vector per horizontal position).
    The BiLSTM processes the sequence and outputs per-timestep class probabilities.
    CTC loss handles the alignment between variable-length predictions and labels.
    """

    def __init__(self, num_classes, hidden_size=256, num_lstm_layers=2):
        super().__init__()

        # CNN encoder: ResNet-18 without avgpool and fc
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.cnn = nn.Sequential(
            backbone.conv1,    # 64 channels
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,   # 64 channels
            backbone.layer2,   # 128 channels
            backbone.layer3,   # 256 channels
            backbone.layer4,   # 512 channels
        )

        # Adaptive pooling to collapse height to 1, keep width as sequence
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))

        # RNN decoder
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_lstm_layers > 1 else 0,
        )

        # Output projection
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # x: (B, 3, H, W)
        conv = self.cnn(x)               # (B, 512, H', W')
        pooled = self.adaptive_pool(conv) # (B, 512, 1, W')
        seq = pooled.squeeze(2)           # (B, 512, W')
        seq = seq.permute(0, 2, 1)       # (B, W', 512) - sequence of features

        lstm_out, _ = self.lstm(seq)      # (B, W', hidden*2)
        logits = self.fc(lstm_out)        # (B, W', num_classes)

        return logits

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CRNN(NUM_CLASSES, HIDDEN_SIZE, NUM_LSTM_LAYERS).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,} total, {trainable_params:,} trainable')
print(f'Model size: ~{total_params * 4 / 1e6:.0f} MB (fp32)')

---
## 6. Training Loop

In [ ]:
import time
import json
import editdistance
from tqdm import tqdm

def ctc_decode_greedy(logits, idx_to_char, blank_idx=0):
    """Greedy CTC decoding: take argmax per timestep, collapse repeats, remove blanks."""
    indices = logits.argmax(dim=-1)  # (B, T)
    results = []
    for seq in indices:
        chars = []
        prev = None
        for idx in seq:
            idx = idx.item()
            if idx != prev and idx != blank_idx:
                chars.append(idx_to_char.get(idx, ''))
            prev = idx
        results.append(''.join(chars))
    return results

def compute_cer(predictions, references):
    """Compute mean Character Error Rate."""
    total_dist = 0
    total_chars = 0
    for pred, ref in zip(predictions, references):
        total_dist += editdistance.eval(pred, ref)
        total_chars += max(len(ref), 1)
    return total_dist / total_chars if total_chars > 0 else 0.0

print('Training utilities ready')

In [ ]:
# Training
ctc_loss_fn = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'val_exact_match': []}
best_cer = float('inf')

# Load existing history if resuming
history_path = f'{CHECKPOINT_DIR}/history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print(f'Loaded history: {len(history["train_loss"])} prior epochs')
    best_cer = min(history['val_cer']) if history['val_cer'] else float('inf')

start_epoch = len(history['train_loss'])

print(f'\n{"="*60}')
print(f'Training CNN+RNN: {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}')
print(f'{"="*60}\n')

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()

    # --- Train ---
    model.train()
    train_loss = 0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'  Epoch {epoch+1}/{EPOCHS} Train', leave=False)
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label']
        label_lengths = batch['label_length']

        optimizer.zero_grad()

        logits = model(images)              # (B, T, C)
        log_probs = logits.log_softmax(2)   # CTC expects log probs
        log_probs = log_probs.permute(1, 0, 2)  # (T, B, C) for CTC

        input_lengths = torch.full((images.size(0),), log_probs.size(0), dtype=torch.long)

        loss = ctc_loss_fn(log_probs, labels, input_lengths, label_lengths)

        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()
            n_batches += 1

        pbar.set_postfix(loss=f'{train_loss / max(n_batches, 1):.4f}')

    scheduler.step()
    avg_train_loss = train_loss / max(n_batches, 1)

    # --- Validate ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_refs = []
    n_val = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'  Epoch {epoch+1}/{EPOCHS} Val', leave=False):
            images = batch['image'].to(device)
            labels = batch['label']
            label_lengths = batch['label_length']
            texts = batch['text']

            logits = model(images)
            log_probs = logits.log_softmax(2).permute(1, 0, 2)
            input_lengths = torch.full((images.size(0),), log_probs.size(0), dtype=torch.long)

            loss = ctc_loss_fn(log_probs, labels, input_lengths, label_lengths)
            if torch.isfinite(loss):
                val_loss += loss.item()
                n_val += 1

            # Decode predictions
            preds = ctc_decode_greedy(logits, idx_to_char, BLANK_IDX)
            all_preds.extend(preds)
            all_refs.extend(texts)

    avg_val_loss = val_loss / max(n_val, 1)
    cer = compute_cer(all_preds, all_refs)
    exact = sum(1 for p, r in zip(all_preds, all_refs) if p.strip() == r.strip()) / max(len(all_preds), 1)
    elapsed = time.time() - epoch_start

    # Sample predictions
    for i in range(min(3, len(all_preds))):
        print(f"  Sample {i}: ref='{all_refs[i][:30]}' pred='{all_preds[i][:30]}'")

    print(f'Epoch {epoch+1}/{EPOCHS} ({elapsed:.0f}s) | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | CER: {cer:.4f} | Exact: {exact:.1%}')

    # Save history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_cer'].append(cer)
    history['val_exact_match'].append(exact)

    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    # Save best checkpoint
    if cer < best_cer:
        best_cer = cer
        torch.save(model.state_dict(), f'{CHECKPOINT_DIR}/best_model.pt')
        print(f'  New best CER: {best_cer:.4f} -> saved')

print(f'\nTraining complete. Best CER: {best_cer:.4f}')
print(f'History: {history_path}')

---
## 7. Results

In [ ]:
# Load and display training history
import json

with open(f'{CHECKPOINT_DIR}/history.json') as f:
    history = json.load(f)

print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'CER':>10} {'Exact':>8}")
print('-' * 50)
for i in range(len(history['train_loss'])):
    print(f"{i+1:>6} {history['train_loss'][i]:>12.4f} {history['val_loss'][i]:>10.4f} {history['val_cer'][i]:>9.2%} {history['val_exact_match'][i]:>7.1%}")

print(f"\nBest CER: {min(history['val_cer']):.4f} (epoch {history['val_cer'].index(min(history['val_cer'])) + 1})")

In [ ]:
# Comparison table
print('\n' + '=' * 65)
print('MODEL COMPARISON — Stage 1 Synthetic Training')
print('=' * 65)
print(f"{'Model':<25} {'Params':>10} {'Best CER':>10} {'Best Exact':>12}")
print('-' * 65)
print(f"{'CNN+RNN (this run)':<25} {'~15M':>10} {min(history['val_cer']):>9.2%} {max(history['val_exact_match']):>11.1%}")
print(f"{'TrOCR-small frozen':<25} {'62M':>10} {'0.14%':>10} {'99.5%':>12}")
print(f"{'TrOCR-small unfrozen':<25} {'62M':>10} {'0.06%':>10} {'99.8%':>12}")
print(f"{'GPT-4o (zero-shot)':<25} {'~200B':>10} {'98.2%':>10} {'0.0%':>12}")
print(f"{'Claude (few-shot)':<25} {'~100B+':>10} {'80.1%':>10} {'0.0%':>12}")
print('-' * 65)

In [ ]:
# Save final results
results = {
    'model': 'CNN+RNN (ResNet18 + BiLSTM + CTC)',
    'params': sum(p.numel() for p in model.parameters()),
    'best_cer': min(history['val_cer']),
    'best_epoch': history['val_cer'].index(min(history['val_cer'])) + 1,
    'best_exact_match': max(history['val_exact_match']),
    'epochs': len(history['train_loss']),
    'config': {
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'hidden_size': HIDDEN_SIZE,
        'num_lstm_layers': NUM_LSTM_LAYERS,
        'img_size': f'{IMG_HEIGHT}x{IMG_WIDTH}',
    },
    'history': history,
}

with open(f'{CHECKPOINT_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to: {CHECKPOINT_DIR}/results.json')

---

### PHASE 2: Fine-tune on synthetic-freeform

Fine-tune the Phase 1 CNN+RNN checkpoint on 280 synthetic-freeform samples
and evaluate on a held-out test split of 35 samples.


In [ ]:
# ============================================================
# PHASE 2: Load synthetic-freeform data
# ============================================================
import csv, random, shutil
from pathlib import Path

FREEFORM_DRIVE = Path('/content/drive/MyDrive/ogham_ocr/datasets/synthetic_freeform')
LOCAL_FF = Path(REPO_DIR) / 'ogham_dataset' / 'synthetic_freeform'
P2_CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/cnn_rnn_phase2'
os.makedirs(P2_CHECKPOINT_DIR, exist_ok=True)

P2_EPOCHS = 5
P2_LR = 1e-4
P2_BATCH_SIZE = 32

LOCAL_FF.mkdir(parents=True, exist_ok=True)
!rsync -a {FREEFORM_DRIVE}/ {LOCAL_FF}/

# Deterministic 80/10/10 split (seed 42 for reproducibility with TrOCR experiments)
freeform_labels = {}
all_ids = []
with open(LOCAL_FF / 'labels.csv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        idx = int(row['image_file'].replace('freeform_', '').replace('.png', ''))
        freeform_labels[idx] = (row['image_file'], row['ogham_text'])
        all_ids.append(idx)

random.seed(42)
random.shuffle(all_ids)
n = len(all_ids)
n_val = n_test = max(1, n // 10)
n_train = n - n_val - n_test
p2_train_ids = all_ids[:n_train]
p2_val_ids   = all_ids[n_train:n_train + n_val]
p2_test_ids  = all_ids[n_train + n_val:]
print(f'Freeform split: train={len(p2_train_ids)}, val={len(p2_val_ids)}, test={len(p2_test_ids)}')


class FreeformCTCDataset(torch.utils.data.Dataset):
    def __init__(self, indices, transform, labels_map, char_to_idx, max_label_len=64):
        self.indices = indices
        self.transform = transform
        self.labels_map = labels_map
        self.char_to_idx = char_to_idx
        self.max_label_len = max_label_len
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        fname, text = self.labels_map[idx]
        img = Image.open(LOCAL_FF / 'images' / fname).convert('RGB')
        img = self.transform(img)
        text = text.replace('\u1680', ' ')
        label = [self.char_to_idx.get(c, 0) for c in text if c in self.char_to_idx]
        label_length = len(label)
        label = label[:self.max_label_len]
        label_length = min(label_length, self.max_label_len)
        label = label + [0] * (self.max_label_len - len(label))
        return {
            'image': img,
            'label': torch.tensor(label, dtype=torch.long),
            'label_length': torch.tensor(label_length, dtype=torch.long),
            'text': text,
        }


p2_train_loader = DataLoader(
    FreeformCTCDataset(p2_train_ids, train_transform, freeform_labels, char_to_idx),
    batch_size=P2_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
)
p2_val_loader = DataLoader(
    FreeformCTCDataset(p2_val_ids, val_transform, freeform_labels, char_to_idx),
    batch_size=P2_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
)
p2_test_loader = DataLoader(
    FreeformCTCDataset(p2_test_ids, val_transform, freeform_labels, char_to_idx),
    batch_size=P2_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
)
print('Phase 2 data loaders ready')


In [ ]:
# ============================================================
# PRE-PHASE-2 BASELINE: Phase 1 model's performance on freeform test set
# ============================================================
# Load the Phase 1 best checkpoint (pre-fine-tune-on-freeform)
p1_ckpt = f'{CHECKPOINT_DIR}/best_model.pt'
model.load_state_dict(torch.load(p1_ckpt, map_location=device))
model.eval()

refs_pre, preds_pre = [], []
with torch.no_grad():
    for batch in p2_test_loader:
        logits = model(batch['image'].to(device))
        preds_pre += ctc_decode_greedy(logits.cpu(), idx_to_char, BLANK_IDX)
        refs_pre += list(batch['text'])

refs_ns_pre = [r.replace(' ', '') for r in refs_pre]
preds_ns_pre = [p.replace(' ', '') for p in preds_pre]
pre_p2_cer = compute_cer(preds_ns_pre, refs_ns_pre)
pre_p2_exact = sum(1 for p, r in zip(preds_ns_pre, refs_ns_pre) if p == r) / max(len(refs_ns_pre), 1)

print(f'\n{"="*70}')
print(f'CNN+RNN PRE-PHASE-2 BASELINE on synth-freeform TEST ({len(refs_pre)} samples)')
print(f'{"="*70}')
print(f'  Character-level CER: {pre_p2_cer*100:.2f}%')
print(f'  Exact match:         {pre_p2_exact*100:.1f}%')
print(f'{"="*70}')
print('This is the Phase 1 model evaluated on freeform — i.e., what CER')
print('would be WITHOUT Phase 2 fine-tuning. Phase 2 training in the next')
print('cell should improve on this number.')


In [ ]:
# ============================================================
# PHASE 2: Load best Phase 1 checkpoint and fine-tune
# ============================================================
# Seed Phase 2 checkpoint dir from Phase 1 best
p1_ckpt = f'{CHECKPOINT_DIR}/best_model.pt'
p2_ckpt = f'{P2_CHECKPOINT_DIR}/best_model.pt'
if not os.path.exists(p2_ckpt) and os.path.exists(p1_ckpt):
    shutil.copyfile(p1_ckpt, p2_ckpt)
    print(f'Seeded Phase 2 ckpt from Phase 1 best')

# Load checkpoint weights into model
state = torch.load(p2_ckpt, map_location=device)
model.load_state_dict(state)
print('Loaded Phase 1 weights into model')

# Lower-LR fine-tune on freeform only (pure fine-tune, no curriculum mix)
p2_optimizer = torch.optim.Adam(model.parameters(), lr=P2_LR)

p2_history = {'train_loss': [], 'val_cer': [], 'val_exact_match': []}
p2_best_cer = float('inf')

print(f'\n{"="*60}')
print(f'Phase 2 CNN+RNN fine-tune: {P2_EPOCHS} epochs, batch {P2_BATCH_SIZE}, lr {P2_LR}')
print(f'{"="*60}\n')

for epoch in range(P2_EPOCHS):
    epoch_start = time.time()
    model.train()
    train_loss, n_batches = 0, 0
    for batch in tqdm(p2_train_loader, desc=f'  Epoch {epoch+1}/{P2_EPOCHS}', leave=False):
        images = batch['image'].to(device)
        labels = batch['label']
        label_lengths = batch['label_length']
        p2_optimizer.zero_grad()
        logits = model(images)          # (B, T, num_classes)
        log_probs = logits.log_softmax(2).transpose(0, 1)  # (T, B, C) for CTC
        input_lengths = torch.full((images.size(0),), logits.size(1), dtype=torch.long)
        loss = ctc_loss_fn(log_probs, labels, input_lengths, label_lengths)
        if not torch.isfinite(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        p2_optimizer.step()
        train_loss += loss.item()
        n_batches += 1
    avg_train_loss = train_loss / max(n_batches, 1)

    # Val (character-level CER, space-stripped)
    model.eval()
    refs, preds = [], []
    with torch.no_grad():
        for batch in p2_val_loader:
            logits = model(batch['image'].to(device))
            preds += ctc_decode_greedy(logits.cpu(), idx_to_char, BLANK_IDX)
            refs += list(batch['text'])
    refs_ns = [r.replace(' ', '') for r in refs]
    preds_ns = [p.replace(' ', '') for p in preds]
    val_cer = compute_cer(preds_ns, refs_ns)
    val_exact = sum(1 for p, r in zip(preds_ns, refs_ns) if p == r) / max(len(refs_ns), 1)

    p2_history['train_loss'].append(avg_train_loss)
    p2_history['val_cer'].append(val_cer)
    p2_history['val_exact_match'].append(val_exact)

    if val_cer < p2_best_cer:
        p2_best_cer = val_cer
        torch.save(model.state_dict(), p2_ckpt)

    print(f'Epoch {epoch+1}/{P2_EPOCHS} ({time.time()-epoch_start:.0f}s) | '
          f'Train Loss: {avg_train_loss:.4f} | Val CER (no-sp): {val_cer*100:.2f}% | '
          f'Val Exact: {val_exact*100:.1f}%')

with open(f'{P2_CHECKPOINT_DIR}/history.json', 'w') as f:
    json.dump(p2_history, f, indent=2)
print(f'\nBest Phase 2 val CER: {p2_best_cer*100:.2f}%')


In [ ]:
# ============================================================
# PHASE 2: Evaluate on pristine test split
# ============================================================
# Load best Phase 2 checkpoint
model.load_state_dict(torch.load(p2_ckpt, map_location=device))
model.eval()

refs, preds = [], []
with torch.no_grad():
    for batch in p2_test_loader:
        logits = model(batch['image'].to(device))
        preds += ctc_decode_greedy(logits.cpu(), idx_to_char, BLANK_IDX)
        refs += list(batch['text'])

# Character-level metrics (space-stripped)
refs_ns = [r.replace(' ', '') for r in refs]
preds_ns = [p.replace(' ', '') for p in preds]
test_cer = compute_cer(preds_ns, refs_ns)
test_exact = sum(1 for p, r in zip(preds_ns, refs_ns) if p == r) / max(len(refs_ns), 1)

print(f'\n{"="*70}')
print(f'CNN+RNN Phase 2 — TEST SPLIT ({len(refs)} pristine samples)')
print(f'{"="*70}')
print(f'  Character-level CER: {test_cer*100:.2f}%')
print(f'  Exact match:         {test_exact*100:.1f}% ({sum(1 for p,r in zip(preds_ns, refs_ns) if p==r)}/{len(refs_ns)})')
print(f'{"="*70}')

# Save results
p2_results = {
    'model': 'CNN+RNN (ResNet18 + BiLSTM + CTC) — Phase 2',
    'training': {'epochs': P2_EPOCHS, 'lr': P2_LR, 'batch_size': P2_BATCH_SIZE,
                 'train_samples': len(p2_train_ids), 'val_samples': len(p2_val_ids)},
    'test': {'n_samples': len(refs), 'cer_pct': test_cer * 100,
             'exact_match_pct': test_exact * 100,
             'metric_note': 'CER and exact match computed on Ogham character sequences with whitespace excluded'},
    'history': p2_history,
    'per_sample': [{'ref': r, 'pred': p} for r, p in zip(refs, preds)],
}
with open(f'{P2_CHECKPOINT_DIR}/phase2_test_results.json', 'w') as f:
    json.dump(p2_results, f, indent=2, ensure_ascii=False)
print(f'\nResults saved to {P2_CHECKPOINT_DIR}/phase2_test_results.json')
